# Muhtemel Ask - 04 ARCHIVE V2
Run this notebook only after legacy `02_FINALIZE.ipynb` or `03_FINALIZE_V2.ipynb` reports **FINALIZATION PASS**. It loads code only from the isolated `SYSTEM_V2_BETA` tree, accepts the legacy version-1 report `*_FINALIZATION_REPORT.json` for migration, and prefers the isolated version-2 report `*_FINALIZATION_REPORT_V2.json` when present.

The completed archive contains exactly three episode files:

- `final/Muhtemel Ask X.Bolum.mkv`
- `final/subtitles/Muhtemel Ask X.Bolum-id.srt`
- `final/subtitles/Muhtemel Ask X.Bolum-tr.srt`

The MKV embeds Indonesian and Turkish tracks, so Infuse does not need duplicate sidecars. The standalone SRTs remain once in `final/subtitles/` for editing or export.

A source video and a final MKV may both exist before cleanup. That temporary duplication preserves resumability while stream-copy identity and subtitle round-trip checks run. The source and work files are removed only after every verification passes, a READY receipt is written outside the episode, and you type the exact episode-bound confirmation phrase.

In [ ]:
EPISODE = 11  # @param {type:"integer"}
DELETE_INTERMEDIATES = False  # @param {type:"boolean"}

if isinstance(EPISODE, bool) or not isinstance(EPISODE, int) or EPISODE < 1:
    raise ValueError("EPISODE must be a positive integer")
if not isinstance(DELETE_INTERMEDIATES, bool):
    raise ValueError("DELETE_INTERMEDIATES must be True or False")

## Mount Drive and prepare FFmpeg
The archive verifier reads the compressed A/V packets and extracts both embedded subtitle tracks. It never re-encodes video or audio.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import importlib
import shutil
import subprocess
import sys

SYSTEM_ROOT = Path("/content/drive/MyDrive/Muhtemel_Ask_Subtitles/SYSTEM_V2_BETA")
if not (SYSTEM_ROOT / "src/archive_v2.py").is_file():
    raise FileNotFoundError(
        f"Archive V2 module was not found at {SYSTEM_ROOT}. Upload the current SYSTEM_V2_BETA release."
    )
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
if str(SYSTEM_ROOT) not in sys.path:
    sys.path.insert(0, str(SYSTEM_ROOT))
for _module_name in list(sys.modules):
    if _module_name == "src" or _module_name.startswith("src."):
        del sys.modules[_module_name]
importlib.invalidate_caches()
print("Runtime ready.")

## Resolve the exact episode and V2 receipt
The module detects the legacy version-1 or isolated version-2 FINALIZATION report automatically. The different report names preserve V1 rollback during a V2 pilot. The V2 receipt is stored in `ARCHIVE_REPORTS/`, outside the episode, so an interrupted or completed cleanup remains independently resumable after the in-episode report is removed.

In [ ]:
import src.archive_v2 as archive_v2

_module_path = Path(archive_v2.__file__).resolve()
if SYSTEM_ROOT.resolve() not in _module_path.parents:
    raise RuntimeError(f"Loaded Archive V2 module from an unexpected path: {_module_path}")
archive_episode_v2 = archive_v2.archive_episode_v2
canonical_archive_paths = archive_v2.canonical_archive_paths
expected_cleanup_confirmation_v2 = archive_v2.expected_cleanup_confirmation_v2

EPISODE_NAME = f"Muhtemel Ask {EPISODE}.Bolum"
DRIVE_ROOT = Path("/content/drive/MyDrive/Muhtemel_Ask_Subtitles")
EPISODE_ROOT = DRIVE_ROOT / "EPISODES" / EPISODE_NAME
RECEIPT_PATH = (
    DRIVE_ROOT / "ARCHIVE_REPORTS" / f"{EPISODE_NAME}_ARCHIVE_RECEIPT_V2.json"
)
if not EPISODE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Episode workspace not found: {EPISODE_ROOT}. Run PREPARE and FINALIZE first."
    )
RETAINED_PATHS = canonical_archive_paths(EPISODE_ROOT, EPISODE)
EXPECTED_CONFIRMATION = expected_cleanup_confirmation_v2(EPISODE)

print(f"Episode: {EPISODE_ROOT}")
print(f"Receipt: {RECEIPT_PATH}")
print("Exact retained layout after verified cleanup:")
for _label in ("mkv", "id_srt", "tr_srt"):
    print(f"  KEEP {RETAINED_PATHS[_label].relative_to(EPISODE_ROOT)}")
if not DELETE_INTERMEDIATES:
    print("Dry-run mode: source/work duplicates will remain for resumability.")
else:
    print(f"Cleanup requires this exact phrase: {EXPECTED_CONFIRMATION}")

## Verify, preview, then optionally clean
`DELETE_INTERMEDIATES = False` is the default dry run. It creates or validates the canonical archive, writes the external READY receipt, and returns the exact deletion preview without deleting source or work files.

When cleanup is enabled, the input prompt is reached only from inside `archive_episode_v2`, after canonical MKV stream hashes, both subtitle round trips, the protected-file whitelist, and the exact cleanup snapshot have passed. Any unexpected video or filesystem change stops the operation.

In [ ]:
def request_cleanup_confirmation(expected):
    if expected != EXPECTED_CONFIRMATION:
        raise RuntimeError("Archive module returned an unexpected confirmation phrase")
    print("WARNING: verified cleanup permanently removes episode source/work duplicates.")
    print("Only the canonical MKV and two SRT files shown above will remain.")
    return input(f"Type exactly '{expected}' to continue: ")

archive_result = archive_episode_v2(
    episode_root=EPISODE_ROOT,
    receipt_path=RECEIPT_PATH,
    episode=EPISODE,
    delete_intermediates=DELETE_INTERMEDIATES,
    confirmation=request_cleanup_confirmation if DELETE_INTERMEDIATES else None,
)

## Result
A successful dry run leaves source/work files in place by design. A successful cleanup leaves exactly the three displayed files in the episode folder. If deleted Drive files still appear in Trash and count toward quota, inspect and empty Trash manually; this notebook never empties unrelated Drive Trash.

In [ ]:
from pprint import pprint

pprint(archive_result, sort_dicts=False)
print("")
_pilot_warnings = archive_result.get("pilot_warnings", [])
if _pilot_warnings:
    print("PILOT REVIEW WARNINGS (cleanup may proceed, but spot-check these):")
    for _warning in _pilot_warnings:
        print(f"  WARNING: {_warning.get('message', _warning)}")
    print("")
for _label in ("mkv", "id_srt", "tr_srt"):
    _path = RETAINED_PATHS[_label]
    print(f"{_label}: {'PRESENT' if _path.is_file() else 'MISSING'} - {_path}")

if DELETE_INTERMEDIATES:
    _expected = {path.resolve() for path in RETAINED_PATHS.values()}
    _actual = {path.resolve() for path in EPISODE_ROOT.rglob("*") if path.is_file()}
    if _actual != _expected:
        raise RuntimeError(
            "Archive V2 cleanup did not leave exactly the canonical MKV and two SRT files"
        )
    print("ARCHIVE V2 CLEANUP PASS - exactly three canonical files remain.")
else:
    print("ARCHIVE V2 DRY RUN PASS - no source/work cleanup was requested.")
    print("Set DELETE_INTERMEDIATES=True and rerun when you are ready to reclaim space.")